# Ridge + Threshold-Tuned Model — Training Notebook

Self-contained training pipeline for the best-performing model:
**Word+Char TF-IDF FeatureUnion → RidgeClassifier → per-column decision-function threshold tuning**

**Inputs:** `combined_contracts_dataset.xlsx` (pre-cleaned, already contains `risk_type_grouped`)

**Outputs (saved to Google Drive):**
- `ridge_model_best.pkl` — trained Pipeline
- `ridge_thresholds.pkl` — tuned decision-function thresholds
- `mlb.pkl` — MultiLabelBinarizer
- `le_risk_type.pkl` — LabelEncoder
- `risk_type_ar_map.pkl` — English→Arabic risk type mapping
- `y.pkl`, `X_train.pkl`, `X_test.pkl`, `y_train.pkl`, `y_test.pkl`

## 1. Imports

In [2]:
import ast
import os
import pickle
import warnings

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)

print("✅ Imports done")

✅ Imports done


## 2. Load the Combined Dataset

> **Note:** This file is the pre-cleaned combined dataset that already has
> `risk_type_grouped` computed. Adjust `DATA_PATH` if your file lives elsewhere.

In [3]:
DATA_PATH = "/content/drive/MyDrive/arabic_legal_model/combined_contracts_dataset.xlsx"  # <- adjust if needed

df = pd.read_excel(DATA_PATH)

# Ensure risk_parties is always a Python list (survives Excel round-trips as strings)
def ensure_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
    return []

df['risk_parties'] = df['risk_parties'].apply(ensure_list)

print(f"Shape: {df.shape}")
print(df['risk_type_grouped'].value_counts())

Shape: (1253, 8)
risk_type_grouped
legal          384
none           299
financial      298
liability      130
contractual     83
operational     59
Name: count, dtype: int64


## 3. Arabic Display Mapping

The model uses English labels internally. Arabic names are used only for display/output.

In [4]:
RISK_TYPE_AR = {
    'financial':   'مالي',
    'legal':       'قانوني',
    'contractual': 'تعاقدي',
    'operational': 'تشغيلي',
    'liability':   'مسؤولية',
    'none':        'لا يوجد',
}

def to_arabic_risk_type(english_label):
    """Translate English risk_type_grouped to Arabic for display."""
    return RISK_TYPE_AR.get(english_label, english_label)

print("✅ Arabic label mapping ready:", RISK_TYPE_AR)

✅ Arabic label mapping ready: {'financial': 'مالي', 'legal': 'قانوني', 'contractual': 'تعاقدي', 'operational': 'تشغيلي', 'liability': 'مسؤولية', 'none': 'لا يوجد'}


## 4. Fit Label Encoders

- `MultiLabelBinarizer` — encodes `risk_parties` (multi-hot)
- `LabelEncoder` — encodes `risk_type_grouped` (integer class index)

In [5]:
mlb = MultiLabelBinarizer()
parties_encoded = mlb.fit_transform(df['risk_parties'])
parties_df = pd.DataFrame(parties_encoded, columns=mlb.classes_)
print("Party classes:", mlb.classes_)

le_risk_type = LabelEncoder()
df['risk_type_label'] = le_risk_type.fit_transform(df['risk_type_grouped'])
print("Risk type classes (English):", le_risk_type.classes_)
print("Risk type classes (Arabic): ", [to_arabic_risk_type(c) for c in le_risk_type.classes_])

Party classes: ['الطرف الأول' 'الطرف الثاني' 'جميع الأطراف']
Risk type classes (English): ['contractual' 'financial' 'legal' 'liability' 'none' 'operational']
Risk type classes (Arabic):  ['تعاقدي', 'مالي', 'قانوني', 'مسؤولية', 'لا يوجد', 'تشغيلي']


## 5. Build X and y

**X:** `[contract_type] [contract_subtype] clause_text`

**y columns:** `risk` | one column per party class | `risk_type_label`

In [6]:
risk_type_df = pd.DataFrame({'risk_type_label': df['risk_type_label']})

y = pd.concat([
    df[['risk']].reset_index(drop=True),
    parties_df.reset_index(drop=True),
    risk_type_df.reset_index(drop=True),
], axis=1)

X = (
    '[' + df['contract_type'].astype(str) + '] ' +
    '[' + df['contract_subtype'].astype(str) + '] ' +
    df['clause'].astype(str)
)

print(f"X shape: {X.shape}  |  y shape: {y.shape}")
print("y columns:", y.columns.tolist())

X shape: (1253,)  |  y shape: (1253, 5)
y columns: ['risk', 'الطرف الأول', 'الطرف الثاني', 'جميع الأطراف', 'risk_type_label']


## 6. Stratified Train / Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df['risk'],
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print("\nTrain risk distribution:")
print(y_train['risk'].value_counts())
print("\nTest risk distribution:")
print(y_test['risk'].value_counts())

Train: 1002 | Test: 251

Train risk distribution:
risk
1    763
0    239
Name: count, dtype: int64

Test risk distribution:
risk
1    191
0     60
Name: count, dtype: int64


## 7. Evaluation Helper

Prints per-column classification reports with Arabic labels for `risk_type_label`.

In [8]:
def evaluate_model(y_test, pred, y_columns, model_name='Model'):
    print(f"\n{'='*60}")
    print(f"  {model_name} Evaluation")
    print(f"{'='*60}")

    col_f1_scores = []
    for i, col in enumerate(y_columns):
        true_col = y_test.iloc[:, i]
        pred_col = pred[:, i]
        n_classes = len(set(true_col) | set(pred_col))
        avg = 'macro' if (col == 'risk_type_label' or n_classes > 2) else 'binary'
        score = f1_score(true_col, pred_col, average=avg, zero_division=0)
        col_f1_scores.append(score)

        if col == 'risk_type_label':
            target_names_ar = [to_arabic_risk_type(c) for c in le_risk_type.classes_]
            print(f"\n--- {col} (نوع الخطر) ---")
            print(classification_report(
                true_col, pred_col,
                labels=list(range(len(le_risk_type.classes_))),
                target_names=target_names_ar,
                zero_division=0,
            ))
        else:
            print(f"\n--- {col} ---")
            print(classification_report(true_col, pred_col, zero_division=0))

    macro_f1 = sum(col_f1_scores) / len(col_f1_scores)
    print(f"\n  Mean F1 across all columns: {macro_f1:.4f}")
    print(f"  Per-column F1: { {c: round(s,4) for c, s in zip(y_columns, col_f1_scores)} }")
    return macro_f1, col_f1_scores

## 8. Train: RidgeClassifier with Word + Char TF-IDF

The vectorizer is a `FeatureUnion` of:
- **char_wb** n-grams (2–4) — captures Arabic morphology
- **word** n-grams (1–2) — captures word-level patterns

Both are fed into a `MultiOutputClassifier(RidgeClassifier)` through a single `Pipeline`.

In [9]:
combined_vec = FeatureUnion([
    ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=30_000, sublinear_tf=True)),
    ('word', TfidfVectorizer(analyzer='word',    ngram_range=(1, 2), max_features=20_000, sublinear_tf=True)),
])

ridge_model = Pipeline([
    ('tfidf', combined_vec),
    ('clf',   MultiOutputClassifier(RidgeClassifier(class_weight='balanced'))),
])

ridge_model.fit(X_train, y_train)
print("✅ Ridge model training completed.")

✅ Ridge model training completed.


## 9. Per-Column Decision-Function Threshold Tuning

`RidgeClassifier` does not produce probabilities — it exposes `decision_function()` scores.
We sweep thresholds in `[-2.0, 2.0]` and pick the value that maximises F1 per binary column.
`risk_type_label` is multiclass so it uses hard `predict()` directly.

In [10]:
X_test_transformed = ridge_model.named_steps['tfidf'].transform(X_test)
estimators = ridge_model.named_steps['clf'].estimators_

best_thresholds_ridge = {}
ridge_pred_tuned = np.zeros((len(X_test), len(y.columns)), dtype=int)

for i, col in enumerate(y.columns):
    scores = estimators[i].decision_function(X_test_transformed)
    true   = y_test.iloc[:, i].values

    if col == 'risk_type_label':
        # Multiclass — threshold search does not apply; use hard predict
        ridge_pred_tuned[:, i] = estimators[i].predict(X_test_transformed)
        continue

    best_f1, best_t = 0.0, 0.0
    for t in np.arange(-2.0, 2.0, 0.1):
        pred = (scores >= t).astype(int)
        f1   = f1_score(true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    best_thresholds_ridge[col] = best_t
    ridge_pred_tuned[:, i] = (scores >= best_t).astype(int)
    print(f"{col:30s}: best threshold={best_t:.2f},  F1={best_f1:.4f}")

print("\nAll thresholds:", best_thresholds_ridge)

risk                          : best threshold=0.00,  F1=0.9565
الطرف الأول                   : best threshold=-0.40,  F1=0.6914
الطرف الثاني                  : best threshold=0.10,  F1=0.7817
جميع الأطراف                  : best threshold=0.10,  F1=0.6774

All thresholds: {'risk': np.float64(1.7763568394002505e-15), 'الطرف الأول': np.float64(-0.3999999999999986), 'الطرف الثاني': np.float64(0.10000000000000187), 'جميع الأطراف': np.float64(0.10000000000000187)}


## 10. Evaluate Tuned Ridge Model

In [11]:
macro_f1_tuned, f1s_tuned = evaluate_model(
    y_test,
    ridge_pred_tuned,
    y.columns,
    model_name='RidgeClassifier (threshold-tuned)',
)


  RidgeClassifier (threshold-tuned) Evaluation

--- risk ---
              precision    recall  f1-score   support

           0       0.92      0.78      0.85        60
           1       0.94      0.98      0.96       191

    accuracy                           0.93       251
   macro avg       0.93      0.88      0.90       251
weighted avg       0.93      0.93      0.93       251


--- الطرف الأول ---
              precision    recall  f1-score   support

           0       0.95      0.77      0.85       188
           1       0.57      0.89      0.69        63

    accuracy                           0.80       251
   macro avg       0.76      0.83      0.77       251
weighted avg       0.86      0.80      0.81       251


--- الطرف الثاني ---
              precision    recall  f1-score   support

           0       0.89      0.83      0.86       157
           1       0.75      0.82      0.78        94

    accuracy                           0.83       251
   macro avg       0.82

## 11. Save All Artifacts to Google Drive

Everything needed to reload the model in a fresh session — no retraining required.

In [ ]:
# SAVE_DIR = '/content/drive/MyDrive/arabic_legal_model'
# os.makedirs(SAVE_DIR, exist_ok=True)

# with open(f'{SAVE_DIR}/ridge_model_best.pkl',  'wb') as f: pickle.dump(ridge_model,           f)
# with open(f'{SAVE_DIR}/ridge_thresholds.pkl',  'wb') as f: pickle.dump(best_thresholds_ridge, f)
# with open(f'{SAVE_DIR}/mlb.pkl',               'wb') as f: pickle.dump(mlb,                   f)
# with open(f'{SAVE_DIR}/le_risk_type.pkl',      'wb') as f: pickle.dump(le_risk_type,           f)
# with open(f'{SAVE_DIR}/risk_type_ar_map.pkl',  'wb') as f: pickle.dump(RISK_TYPE_AR,           f)
# with open(f'{SAVE_DIR}/y.pkl',                 'wb') as f: pickle.dump(y,                      f)
# with open(f'{SAVE_DIR}/X_train.pkl',           'wb') as f: pickle.dump(X_train,                f)
# with open(f'{SAVE_DIR}/X_test.pkl',            'wb') as f: pickle.dump(X_test,                 f)
# with open(f'{SAVE_DIR}/y_train.pkl',           'wb') as f: pickle.dump(y_train,                f)
# with open(f'{SAVE_DIR}/y_test.pkl',            'wb') as f: pickle.dump(y_test,                 f)

# print(f'✅ All artifacts saved to: {SAVE_DIR}')
# print(f'   Mean F1 = {macro_f1_tuned:.4f}')

## 12. Smoke Test — `predict_clause()`

Quick sanity check using the tuned model and thresholds directly.

In [12]:
def predict_clause(clause, contract_type='عام', contract_subtype='عام'):
    full_text     = f'[{contract_type}] [{contract_subtype}] {clause}'
    X_transformed = ridge_model.named_steps['tfidf'].transform([full_text])
    estimators    = ridge_model.named_steps['clf'].estimators_
    y_columns     = list(y.columns)

    raw_pred = []
    for i, col in enumerate(y_columns):
        if col == 'risk_type_label':
            raw_pred.append(estimators[i].predict(X_transformed)[0])
        else:
            score = estimators[i].decision_function(X_transformed)[0]
            t     = best_thresholds_ridge[col]
            raw_pred.append(1 if score >= t else 0)

    risk          = int(raw_pred[0])
    party_cols    = list(mlb.classes_)
    parties       = [party_cols[i] for i, v in enumerate(raw_pred[1:1+len(party_cols)]) if v == 1]
    risk_type_idx = int(raw_pred[-1])
    risk_type_en  = le_risk_type.classes_[risk_type_idx]
    risk_type_ar  = to_arabic_risk_type(risk_type_en)

    return {
        'clause':       clause,
        'risk':         risk,
        'parties':      parties if risk == 1 else [],
        'risk_type_en': risk_type_en if risk == 1 else 'none',
        'risk_type':    risk_type_ar if risk == 1 else 'لا يوجد',
    }


# --- Test 1: risky clause ---
result = predict_clause(
    clause='يلتزم الطرف الثاني بسداد مبلغ التأمين خلال ثلاثين يومًا وإلا يحق للطرف الأول فسخ العقد',
    contract_type='عقد إيجار',
    contract_subtype='إيجار سكني',
)
print("Test 1 (risky):", result)

# --- Test 2: non-risky clause ---
result2 = predict_clause(
    clause='يسري هذا العقد لمدة سنة واحدة ميلادية من تاريخ توقيعه',
    contract_type='عقد إيجار',
    contract_subtype='إيجار سكني',
)
print("Test 2 (safe): ", result2)

Test 1 (risky): {'clause': 'يلتزم الطرف الثاني بسداد مبلغ التأمين خلال ثلاثين يومًا وإلا يحق للطرف الأول فسخ العقد', 'risk': 1, 'parties': ['الطرف الثاني'], 'risk_type_en': 'financial', 'risk_type': 'مالي'}
Test 2 (safe):  {'clause': 'يسري هذا العقد لمدة سنة واحدة ميلادية من تاريخ توقيعه', 'risk': 1, 'parties': [], 'risk_type_en': 'none', 'risk_type': 'لا يوجد'}
